In [9]:
import requests, re, time
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

OUTPUT_FILE = "fear_greed_log.csv"

def get_fear_greed():
    url = "https://feargreedmeter.com/"
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    text = BeautifulSoup(html, "html.parser").get_text("\n")

    # Index value
    m_val = re.search(r"Stock Market\s*Crypto\s*(\d{1,3})\s*[\r\n]+\s*[+\-]?\d+\s*points", text, re.I | re.S)
    value = m_val.group(1) if m_val else None

    # "Updated X minutes ago" string
    m_updated = re.search(r"(\b(?:\d+\s+)?(?:minute|hour|day)s?\s+ago\b|a\s+minute\s+ago)", text, re.I)
    updated = m_updated.group(1) if m_updated else None

    return value, updated

def log_if_changed(last_value):
    value, updated = get_fear_greed()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if value and value != last_value:
        row = {"CheckedAt": now, "Value": value, "SiteUpdated": updated}
        try:
            df = pd.read_csv(OUTPUT_FILE)
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        except FileNotFoundError:
            df = pd.DataFrame([row])

        df.to_csv(OUTPUT_FILE, index=False)
        print(f"[{now}] Logged new value: {value} (site says updated {updated})")
        return value
    else:
        print(f"[{now}] No change (current: {value}, site says {updated})")
        return last_value

if __name__ == "__main__":
    last_value = None
    while True:
        last_value = log_if_changed(last_value)
        time.sleep(300)  # check every 5 min


[2025-09-11 13:30:48] Logged new value: 53 (site says updated 16 minutes ago)


KeyboardInterrupt: 

In [15]:
import requests, re, time
from bs4 import BeautifulSoup
from datetime import datetime
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# --- Google Sheets Setup ---
scope = ["https://spreadsheets.google.com/feeds","https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("fear-and-greed-471817-146551f43610.json", scope)
client = gspread.authorize(creds)
sheet = client.open("FearAndGreed").sheet1  # change to your sheet name

def get_fear_greed():
    url = "https://feargreedmeter.com/"
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    text = BeautifulSoup(html, "html.parser").get_text("\n")

    m_val = re.search(r"Stock Market\s*Crypto\s*(\d{1,3})\s*[\r\n]+\s*[+\-]?\d+\s*points", text, re.I | re.S)
    value = m_val.group(1) if m_val else None

    m_updated = re.search(r"(\b(?:\d+\s+)?(?:minute|hour|day)s?\s+ago\b|a\s+minute\s+ago)", text, re.I)
    updated = m_updated.group(1) if m_updated else None

    return value, updated

last_value = None
while True:
    value, updated = get_fear_greed()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if value and value != last_value:
        sheet.append_row([now, value, updated])
        print(f"[{now}] Logged to Google Sheets: {value} (site updated {updated})")
        last_value = value
    else:
        print(f"[{now}] No change (current: {value}, site says {updated})")

    time.sleep(60)  # check every 5 min


[2025-09-11 13:48:44] Logged to Google Sheets: 53 (site updated 16 minutes ago)
[2025-09-11 13:49:45] No change (current: 53, site says 16 minutes ago)
[2025-09-11 13:50:45] No change (current: 53, site says 21 minutes ago)
[2025-09-11 13:51:45] No change (current: 53, site says 21 minutes ago)
[2025-09-11 13:52:45] No change (current: 53, site says 21 minutes ago)
[2025-09-11 13:53:46] No change (current: 53, site says 21 minutes ago)
[2025-09-11 13:54:46] No change (current: 53, site says 21 minutes ago)
[2025-09-11 13:55:46] No change (current: 53, site says 16 minutes ago)
[2025-09-11 13:56:46] No change (current: 53, site says 16 minutes ago)
[2025-09-11 13:57:47] No change (current: 53, site says 16 minutes ago)
[2025-09-11 13:58:47] No change (current: 53, site says 16 minutes ago)
[2025-09-11 13:59:47] No change (current: 53, site says 16 minutes ago)
[2025-09-11 14:00:47] No change (current: 53, site says 21 minutes ago)
[2025-09-11 14:01:47] No change (current: 53, site says 

KeyboardInterrupt: 

In [7]:
import requests, re
from bs4 import BeautifulSoup

URL = "https://feargreedmeter.com/"

html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
text = BeautifulSoup(html, "html.parser").get_text("\n")

# 1) Extract the index value that sits between "Stock Market / Crypto" and "points"
m_val = re.search(r"Stock Market\s*Crypto\s*(\d{1,3})\s*[\r\n]+\s*[+\-]?\d+\s*points", text, re.I | re.S)

# 2) (Optional) Extract the site's "Updated … ago" text
m_updated = re.search(r"(\b(?:\d+\s+)?(?:minute|hour|day)s?\s+ago\b|a\s+minute\s+ago)", text, re.I)

print("Value:", m_val.group(1) if m_val else "N/A")
print("Updated:", m_updated.group(1) if m_updated else "N/A")

# If it fails, dump a small snippet to inspect:
if not m_val:
    print("\n--- First 800 chars of page text ---\n")
    print(text[:800])


Value: 53
Updated: 11 minutes ago
